<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS15_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS15 — Pipeline Machine Learning & Deployment Model RumahPoint

Jobsheet asli menutup dengan pipeline dan deployment model klasifikasi siang/malam
(Flask + Docker + Hugging Face Spaces). Versi PBL ini melakukan hal yang sama untuk
proyek RumahPoint:

1. **Pipeline** — semua langkah praproses (`SimpleImputer`, `StandardScaler`,
   `OneHotEncoder`) dan model dibungkus dalam satu objek `Pipeline` scikit-learn.
2. **Deployment** — model disimpan dengan `joblib`, lalu disajikan lewat API Flask
   dengan endpoint `/predict` yang mengembalikan **harga prediksi** dan **kategori**.

Pipeline mencegah *data leakage*: statistik imputasi/scaling dihitung hanya dari data
latih, dan objek yang sama dipakai saat inference.

## Langkah 1 — Data Mentah dan Definisi Kolom

Berbeda dengan notebook sebelumnya, di sini **pembersihan tidak dilakukan manual**.
Pipeline menerima data mentah dan menangani nilai hilang serta encoding sendiri, supaya
persis seperti kondisi aplikasi saat menerima input pengguna.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import json
import time
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, classification_report)

RANDOM_STATE = 42
DATA_PATH = 'dataset/properti_multi_kota.csv'
ARTEFAK = Path('deployment')
ARTEFAK.mkdir(exist_ok=True)

NUM = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']
CAT = ['lokasi', 'kondisi']

df = pd.read_csv(DATA_PATH).drop_duplicates()
df = df[(df['luas_m2'] >= 10) & (df['luas_m2'] <= 2000)]
df = df[df['luas_bangunan'].isna() |
        ((df['luas_bangunan'] >= 10) & (df['luas_bangunan'] <= 2000))]
df = df[df['harga'].between(50_000_000, 100_000_000_000)]
df = df[df['jumlah_kamar'].isna() | (df['jumlah_kamar'] <= 20)]
df = df[df['jumlah_kamar_mandi'].isna() | (df['jumlah_kamar_mandi'] <= 20)]

tahun = df['usia_bangunan'] > 100
df.loc[tahun, 'usia_bangunan'] = 2026 - df.loc[tahun, 'usia_bangunan']
df.loc[~df['usia_bangunan'].between(0, 100) & df['usia_bangunan'].notna(),
       'usia_bangunan'] = np.nan
df['kondisi'] = np.where(df['usia_bangunan'].isna(), 'Tidak diketahui',
                 np.where(df['usia_bangunan'] <= 3, 'Baru', 'Bekas'))

q1, q2 = df['harga'].quantile([1 / 3, 2 / 3])
df['kelas_harga'] = np.where(df['harga'] < q1, 'Murah',
                     np.where(df['harga'] <= q2, 'Sedang', 'Mahal'))
df['log_harga'] = np.log10(df['harga'])

X = df[NUM + CAT]
y_reg = df['log_harga']
y_kls = df['kelas_harga']

X_tr, X_te, yr_tr, yr_te, yk_tr, yk_te = train_test_split(
    X, y_reg, y_kls, test_size=0.2, random_state=RANDOM_STATE, stratify=y_kls)

print(f'Data: {len(df):,} | Train: {len(X_tr):,} | Test: {len(X_te):,}')
print('Kolom input:', NUM + CAT)

Data: 56,170 | Train: 44,936 | Test: 11,234
Kolom input: ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi', 'lokasi', 'kondisi']


## Langkah 2 — Definisi Pipeline

`ColumnTransformer` memisahkan perlakuan untuk kolom numerik dan kategorikal:

| Kolom | Perlakuan |
|---|---|
| `luas_m2`, `luas_bangunan`, `jumlah_kamar`, `jumlah_kamar_mandi` | median imputation → StandardScaler |
| `lokasi`, `kondisi` | OneHotEncoder; kategori langka digabung (`min_frequency=30`), kategori baru saat inference diabaikan |

Dua pipeline dibangun dengan praproses identik: satu untuk regresi harga, satu untuk
klasifikasi kategori.

In [2]:
def buat_preprocessor():
    numerik = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    kategorikal = OneHotEncoder(handle_unknown='ignore', min_frequency=30)
    return ColumnTransformer([
        ('num', numerik, NUM),
        ('cat', kategorikal, CAT),
    ])

pipe_reg = Pipeline([
    ('pre', buat_preprocessor()),
    ('model', RandomForestRegressor(n_estimators=120, min_samples_leaf=3,
                                    random_state=RANDOM_STATE, n_jobs=-1)),
])
pipe_kls = Pipeline([
    ('pre', buat_preprocessor()),
    ('model', RandomForestClassifier(n_estimators=120, min_samples_leaf=3,
                                     random_state=RANDOM_STATE, n_jobs=-1,
                                     class_weight='balanced')),
])

t0 = time.perf_counter()
pipe_reg.fit(X_tr, yr_tr)
pipe_kls.fit(X_tr, yk_tr)
print(f'Waktu training kedua pipeline: {time.perf_counter() - t0:.1f} detik')
print('Jumlah fitur setelah praproses:',
      len(pipe_reg.named_steps['pre'].get_feature_names_out()))

Waktu training kedua pipeline: 51.9 detik
Jumlah fitur setelah praproses: 214


## Langkah 3 — Evaluasi Pipeline

Evaluasi dilakukan pada test set yang tidak dipakai saat fitting. Karena pipeline
menyimpan seluruh langkah, evaluasi ini identik dengan kondisi inference.

In [3]:
pred_log = pipe_reg.predict(X_te)
pred_kls = pipe_kls.predict(X_te)

print('=== Regresi harga (log10) ===')
print(f'RMSE : {np.sqrt(mean_squared_error(yr_te, pred_log)):.4f}')
print(f'MAE  : {mean_absolute_error(yr_te, pred_log):.4f}')
print(f'R2   : {r2_score(yr_te, pred_log):.4f}')
print(f'MAE Rupiah : Rp {mean_absolute_error(10 ** yr_te, 10 ** pred_log):,.0f}')
print()
print('=== Klasifikasi kategori ===')
print(f'Akurasi: {accuracy_score(yk_te, pred_kls):.4f}')
print(classification_report(yk_te, pred_kls, digits=4))

=== Regresi harga (log10) ===
RMSE : 0.2021
MAE  : 0.1389
R2   : 0.7983
MAE Rupiah : Rp 592,953,761

=== Klasifikasi kategori ===
Akurasi: 0.7919
              precision    recall  f1-score   support

       Mahal     0.8328    0.8619    0.8471      3744
       Murah     0.8324    0.8251    0.8288      3745
      Sedang     0.7072    0.6887    0.6978      3745

    accuracy                         0.7919     11234
   macro avg     0.7908    0.7919    0.7912     11234
weighted avg     0.7908    0.7919    0.7912     11234



In [4]:
contoh = X_te.head(5).copy()
hasil = contoh.copy()
hasil['harga_aktual'] = (10 ** yr_te.head(5)).map(lambda v: f'Rp {v:,.0f}').values
hasil['harga_prediksi'] = pd.Series(10 ** pred_log[:5]).map(lambda v: f'Rp {v:,.0f}').values
hasil['kategori_aktual'] = yk_te.head(5).values
hasil['kategori_prediksi'] = pred_kls[:5]
hasil

,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,lokasi,kondisi,harga_aktual,harga_prediksi,kategori_aktual,kategori_prediksi
23582,60.0,50.0,1.0,1.0,Serang,Tidak diketahui,"Rp 350,000,000","Rp 234,774,718",Murah,Murah
31643,70.0,50.0,2.0,2.0,Depok,Tidak diketahui,"Rp 990,000,000","Rp 1,172,343,190",Sedang,Mahal
58231,96.0,52.0,2.0,1.0,Kabupaten Demak,Tidak diketahui,"Rp 380,000,000","Rp 489,834,259",Murah,Murah
53201,84.0,36.0,2.0,1.0,Serang,Tidak diketahui,"Rp 320,000,000","Rp 345,867,079",Murah,Murah
33483,103.0,70.0,3.0,2.0,Kabupaten Tangerang,Tidak diketahui,"Rp 2,248,918,363","Rp 1,294,165,151",Mahal,Sedang


## Langkah 4 — Simpan Artefak

Kedua pipeline disimpan sebagai file `.joblib`. Ukurannya perlu diperhatikan karena
Random Forest menyimpan banyak pohon; kompresi `joblib` (`compress=3`) menekan ukuran
tanpa memperlambat loading secara berarti.

In [5]:
path_reg = ARTEFAK / 'model_harga.joblib'
path_kls = ARTEFAK / 'model_kelas.joblib'

joblib.dump(pipe_reg, path_reg, compress=3)
joblib.dump(pipe_kls, path_kls, compress=3)

for p in [path_reg, path_kls]:
    print(f'{p} : {p.stat().st_size / 1e6:.1f} MB')

# bukti bahwa artefak bisa dimuat ulang dan memberi hasil sama
ulang = joblib.load(path_reg)
assert np.allclose(ulang.predict(X_te.head(3)), pred_log[:3])
print('Artefak berhasil dimuat ulang dan prediksi konsisten.')

deployment/model_harga.joblib : 37.8 MB
deployment/model_kelas.joblib : 10.8 MB


Artefak berhasil dimuat ulang dan prediksi konsisten.


## Langkah 5 — Aplikasi Flask

Sel berikut menulis `deployment/app.py`, `requirements.txt`, dan `Dockerfile`.
Endpoint:

- `GET /` → status dan daftar field yang diterima.
- `POST /predict` → JSON `{lokasi, luas_m2, luas_bangunan, jumlah_kamar,
  jumlah_kamar_mandi, kondisi}` → `{harga_prediksi, harga_juta, kategori}`.

Nilai yang tidak dikirim akan diisi default (median/`Tidak diketahui`) oleh pipeline.

In [6]:
%%writefile deployment/app.py
import os
from pathlib import Path

import joblib
import pandas as pd
from flask import Flask, jsonify, request

BASE = Path(__file__).resolve().parent

model_harga = joblib.load(BASE / 'model_harga.joblib')
model_kelas = joblib.load(BASE / 'model_kelas.joblib')

app = Flask(__name__)

KOLOM = ['luas_m2', 'luas_bangunan', 'jumlah_kamar',
         'jumlah_kamar_mandi', 'lokasi', 'kondisi']
DEFAULT = {
    'luas_m2': 100.0,
    'luas_bangunan': 65.0,
    'jumlah_kamar': 2.0,
    'jumlah_kamar_mandi': 2.0,
    'lokasi': 'Tidak diketahui',
    'kondisi': 'Tidak diketahui',
}


@app.get('/')
def index():
    return jsonify({
        'status': 'ok',
        'model': 'RumahPoint Random Forest',
        'field': KOLOM,
        'contoh': {
            'lokasi': 'Bekasi',
            'luas_m2': 120,
            'luas_bangunan': 90,
            'jumlah_kamar': 3,
            'jumlah_kamar_mandi': 2,
            'kondisi': 'Bekas',
        },
    })


@app.post('/predict')
def predict():
    data = request.get_json(force=True, silent=True) or {}
    baris = {k: data.get(k, v) for k, v in DEFAULT.items()}
    frame = pd.DataFrame([baris])

    log_harga = float(model_harga.predict(frame)[0])
    harga = float(10 ** log_harga)
    kategori = str(model_kelas.predict(frame)[0])

    return jsonify({
        'harga_prediksi': round(harga),
        'harga_juta': round(harga / 1e6, 1),
        'kategori': kategori,
        'input': baris,
    })


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=int(os.environ.get('PORT', 7860)))

Overwriting deployment/app.py


In [7]:
%%writefile deployment/requirements.txt
flask==3.1.3
joblib==1.6.0
numpy==2.5.4
pandas==3.1.1
scikit-learn==1.9.1

Overwriting deployment/requirements.txt


In [8]:
%%writefile deployment/Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 7860
CMD ["python", "app.py"]

Overwriting deployment/Dockerfile


## Langkah 6 — Uji API Tanpa Server (Flask `test_client`)

`test_client` memanggil aplikasi langsung di dalam proses, tanpa membuka port. Ini cara
tercepat memverifikasi endpoint sebelum container dijalankan.

In [9]:
import importlib
import sys

sys.path.insert(0, str(ARTEFAK.resolve()))
import app as aplikasi
importlib.reload(aplikasi)

client = aplikasi.app.test_client()
print('GET / ->', client.get('/').get_json()['status'])

contoh_input = [
    {'lokasi': 'Bekasi', 'luas_m2': 60, 'luas_bangunan': 36,
     'jumlah_kamar': 2, 'jumlah_kamar_mandi': 1, 'kondisi': 'Bekas'},
    {'lokasi': 'Bandung', 'luas_m2': 120, 'luas_bangunan': 90,
     'jumlah_kamar': 3, 'jumlah_kamar_mandi': 2, 'kondisi': 'Baru'},
    {'lokasi': 'Jakarta Selatan', 'luas_m2': 300, 'luas_bangunan': 250,
     'jumlah_kamar': 5, 'jumlah_kamar_mandi': 4, 'kondisi': 'Baru'},
    {'lokasi': 'Kota Antah Berantah', 'luas_m2': 100},
]

for payload in contoh_input:
    resp = client.post('/predict', json=payload)
    print(resp.status_code, resp.get_json())

GET / -> ok
200 {'harga_juta': 450.2, 'harga_prediksi': 450235487, 'input': {'jumlah_kamar': 2, 'jumlah_kamar_mandi': 1, 'kondisi': 'Bekas', 'lokasi': 'Bekasi', 'luas_bangunan': 36, 'luas_m2': 60}, 'kategori': 'Murah'}
200 {'harga_juta': 1864.4, 'harga_prediksi': 1864403888, 'input': {'jumlah_kamar': 3, 'jumlah_kamar_mandi': 2, 'kondisi': 'Baru', 'lokasi': 'Bandung', 'luas_bangunan': 90, 'luas_m2': 120}, 'kategori': 'Mahal'}
200 {'harga_juta': 6917.7, 'harga_prediksi': 6917651712, 'input': {'jumlah_kamar': 5, 'jumlah_kamar_mandi': 4, 'kondisi': 'Baru', 'lokasi': 'Jakarta Selatan', 'luas_bangunan': 250, 'luas_m2': 300}, 'kategori': 'Mahal'}
200 {'harga_juta': 672.0, 'harga_prediksi': 671986216, 'input': {'jumlah_kamar': 2.0, 'jumlah_kamar_mandi': 2.0, 'kondisi': 'Tidak diketahui', 'lokasi': 'Kota Antah Berantah', 'luas_bangunan': 65.0, 'luas_m2': 100}, 'kategori': 'Sedang'}


## Langkah 7 — Tugas: Fungsi Prediksi untuk Project Sendiri

Fungsi berikut membungkus pipeline sehingga bisa dipanggil dari notebook atau script
lain tanpa Flask. Ini bentuk yang dipakai tim mobile/web untuk integrasi.

In [10]:
def prediksi_properti(lokasi, luas_m2, luas_bangunan=None,
                      jumlah_kamar=2, jumlah_kamar_mandi=1,
                      kondisi='Tidak diketahui'):
    baris = pd.DataFrame([{
        'lokasi': lokasi,
        'luas_m2': luas_m2,
        'luas_bangunan': luas_bangunan if luas_bangunan is not None else np.nan,
        'jumlah_kamar': jumlah_kamar,
        'jumlah_kamar_mandi': jumlah_kamar_mandi,
        'kondisi': kondisi,
    }])
    harga = 10 ** float(pipe_reg.predict(baris)[0])
    kategori = str(pipe_kls.predict(baris)[0])
    return {'harga_prediksi': round(harga), 'kategori': kategori}


for kota, luas in [('Sleman', 100), ('Surabaya', 150), ('Jakarta Pusat', 200)]:
    print(kota, luas, 'm2 ->', prediksi_properti(kota, luas))

Sleman 100 m2 -> {'harga_prediksi': 439374162, 'kategori': 'Sedang'}


Surabaya 150 m2 -> {'harga_prediksi': 784843189, 'kategori': 'Mahal'}
Jakarta Pusat 200 m2 -> {'harga_prediksi': 645696920, 'kategori': 'Mahal'}


**Analisis JS15:**

- Pipeline membuat alur **fit → simpan → inference** konsisten. Tanpa pipeline, urutan imputasi/scaling/encoding mudah tertukar saat deploy dan menyebabkan prediksi melenceng.
- Setelah praproses, 6 kolom input menjadi **214 fitur** (4 numerik + one-hot `lokasi`/`kondisi` dengan `min_frequency=30`). Ini jauh lebih kecil daripada one-hot semua lokasi di JS03 (592 kolom), tetapi tetap mempertahankan identitas kota yang sering muncul.
- **Regresi**: RMSE log **0,2021**, MAE log 0,1389, **R² 0,7983**, dan MAE Rupiah **Rp593 juta**. Hasil ini sedikit di bawah Random Forest JS07 (R² 0,8235) karena `min_samples_leaf=3` dan representasi one-hot yang lebih besar membuat model lebih konservatif; sebagai kompensasi, pipeline siap dipakai langsung pada data mentah.
- **Klasifikasi**: akurasi **0,7919**, jauh di atas k-NN (0,755) dan MLP (0,735). Peningkatan ini berasal dari **one-hot lokasi**: identitas kota memberi sinyal harga yang tidak tertangkap `lokasi_freq`. Kelas **Sedang** masih paling sulit (F1 0,698 vs 0,847 Mahal dan 0,829 Murah), konsisten dengan semua notebook sebelumnya.
- **Ukuran artefak**: `model_harga.joblib` 37,8 MB dan `model_kelas.joblib` 10,8 MB (total ±48,6 MB). Untuk produksi, pertimbangkan `n_estimators` lebih kecil, pemangkasan `min_samples_leaf`, atau model lebih ringkas (mis. HistGradientBoosting) bila ukuran menjadi masalah.
- `handle_unknown='ignore'` penting: pengguna bisa memasukkan lokasi yang tidak ada di data latih. Tanpa opsi ini, encoder akan error; dengan opsi ini, lokasi baru diabaikan dan prediksi tetap jalan (dengan akurasi lebih rendah).
- Pengujian `test_client` memastikan endpoint mengembalikan **HTTP 200** dan JSON yang benar untuk properti murah, sedang, mahal, maupun input tidak lengkap. Contoh: Bekasi 60 m² → Rp450 juta (Murah); Jakarta Selatan 300 m² → Rp6,92 miliar (Mahal); input hanya `lokasi` + `luas_m2` tetap diproses dengan nilai default.
- Catatan penting: **harga prediksi dan kategori berasal dari dua model terpisah**, sehingga bisa tampak tidak konsisten dengan batas kuantil. Contoh: Sleman 100 m² diprediksi Rp439 juta (di bawah batas Murah Rp483,8 juta) tetapi diklasifikasikan **Sedang**. Untuk aplikasi produksi, kategori sebaiknya diturunkan dari harga prediksi, atau kedua model dikalibrasi bersama.
- Untuk deploy sungguhan (Docker/Hugging Face Spaces): salin folder `deployment/`, jalankan `docker build -t rumahpoint .`, lalu `docker run -p 7860:7860 rumahpoint`. File model `.joblib` **tidak** di-commit karena dapat dibuat ulang dengan menjalankan notebook ini.
